In [84]:
import pandas as pd
import numpy as np
import string
import pm4py
import warnings
import joblib
import torch
import torch.nn as nn

In [112]:
# Fixed set of possible activities: a, b, c, ..., t (20 letters)
ACTIVITIES = list(string.ascii_lowercase[:20])  # ['a', 'b', ..., 't']
ACTIVITIES[:5]

['a', 'b', 'c', 'd', 'e']

In [113]:
# All possible bigrams among these activities (400 combinations, includes self-loops like a->a)
ALL_BIGRAMS = [f"{a}->{b}" for a in ACTIVITIES for b in ACTIVITIES]
ALL_BIGRAMS[:5]

['a->a', 'a->b', 'a->c', 'a->d', 'a->e']

In [114]:
# --- Setup ---
warnings.filterwarnings("ignore")

df = pm4py.read_xes("../data/intra.xes")
df = df.sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)
df[:5]

parsing log, completed traces :: 100%|█████████████████████████████████████████| 2464/2464 [00:00<00:00, 18707.32it/s]


,event:id,concept:name,start_timestamp,time:timestamp,event:duration_min,org:resource,case:concept:name,case:amount,case:region
0,evt_00000001,b,2020-01-01 00:00:00+00:00,2020-01-01 00:18:07.337264+00:00,18.122288,res_03,case_000001,1108.44,region_1
1,evt_00000002,g,2020-01-01 00:24:19.353292+00:00,2020-01-01 01:12:03.738374+00:00,47.739751,res_06,case_000001,1108.44,region_1
2,evt_00000003,h,2020-01-01 01:33:28.051364+00:00,2020-01-01 01:59:39.992804+00:00,26.199024,res_07,case_000001,1108.44,region_1
3,evt_00000004,b,2020-01-01 01:35:18.355889+00:00,2020-01-01 01:58:00.780425+00:00,22.707076,res_03,case_000002,935.89,region_1
4,evt_00000005,g,2020-01-01 02:17:58.411572+00:00,2020-01-01 02:45:01.986501+00:00,27.059582,res_06,case_000002,935.89,region_1


In [115]:
case_col = df['case:concept:name']
activity_col = df['concept:name']

In [116]:
# Position of each event within its case (1-based) and total case length
df['event_pos'] = df.groupby(case_col).cumcount() + 1
df['case_length'] = df.groupby(case_col)['event_pos'].transform('max')
df[:5]

,event:id,concept:name,start_timestamp,time:timestamp,event:duration_min,org:resource,case:concept:name,case:amount,case:region,event_pos,case_length
0,evt_00000001,b,2020-01-01 00:00:00+00:00,2020-01-01 00:18:07.337264+00:00,18.122288,res_03,case_000001,1108.44,region_1,1,3
1,evt_00000002,g,2020-01-01 00:24:19.353292+00:00,2020-01-01 01:12:03.738374+00:00,47.739751,res_06,case_000001,1108.44,region_1,2,3
2,evt_00000003,h,2020-01-01 01:33:28.051364+00:00,2020-01-01 01:59:39.992804+00:00,26.199024,res_07,case_000001,1108.44,region_1,3,3
3,evt_00000004,b,2020-01-01 01:35:18.355889+00:00,2020-01-01 01:58:00.780425+00:00,22.707076,res_03,case_000002,935.89,region_1,1,3
4,evt_00000005,g,2020-01-01 02:17:58.411572+00:00,2020-01-01 02:45:01.986501+00:00,27.059582,res_06,case_000002,935.89,region_1,2,3


In [117]:
activity_dummies = pd.get_dummies(activity_col)
activity_dummies = activity_dummies.reindex(columns=ACTIVITIES, fill_value=0)  # force all 20 columns

activity_counts = activity_dummies.groupby(case_col).cumsum()
activity_freq = activity_counts.div(df['event_pos'], axis=0)  # divide by events-so-far
activity_freq = activity_freq.add_prefix('freq_')

In [118]:
prev_activity = df.groupby(case_col)['concept:name'].shift(1)
bigram = prev_activity + '->' + activity_col  # NaN for first event of each case

bigram_dummies = pd.get_dummies(bigram)
bigram_dummies = bigram_dummies.reindex(columns=ALL_BIGRAMS, fill_value=0)  # force all 400 columns

bigram_counts = bigram_dummies.groupby(case_col).cumsum()

In [119]:
n_transitions = (df['event_pos'] - 1).clip(lower=1)  # avoid div-by-zero for first event
bigram_freq = bigram_counts.div(n_transitions, axis=0)
bigram_freq[df['event_pos'] == 1] = 0  # first event: no transitions yet, force 0 instead of NaN/inf
bigram_freq = bigram_freq.add_prefix('bigram_')

In [120]:
distinct_activities = (activity_counts > 0).astype(int)
distinct_activities = distinct_activities.add_prefix('seen_')

In [121]:
df['progress_ratio'] = df['event_pos'] / df['case_length']

In [122]:
# stack the feature blocks; order matches training: freq_*, bigram_*, seen_*, progress_ratio
features = pd.concat([activity_freq, bigram_freq, distinct_activities, df[['progress_ratio']]], axis=1)
features[:5]

,freq_a,freq_b,freq_c,freq_d,freq_e,freq_f,freq_g,freq_h,freq_i,freq_j,...,seen_l,seen_m,seen_n,seen_o,seen_p,seen_q,seen_r,seen_s,seen_t,progress_ratio
0,0.0,1.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0.333333
1,0.0,0.500000,0.0,0.0,0.0,0.0,0.500000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0.666667
2,0.0,0.333333,0.0,0.0,0.0,0.0,0.333333,0.333333,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1.000000
3,0.0,1.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0.333333
4,0.0,0.500000,0.0,0.0,0.0,0.0,0.500000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0.666667


In [123]:
# --- Prepare data ---
FEATURE_COLS = (
    [f"freq_{a}" for a in ACTIVITIES]
    + [f"bigram_{b}" for b in ALL_BIGRAMS]
    + [f"seen_{a}" for a in ACTIVITIES]
    + ["progress_ratio"]
)
features = features[FEATURE_COLS]  # exact column set + order the saved model was trained with
keys = df[['case:concept:name', 'event:id']]  # kept aside, not fed into the model

print(len(FEATURE_COLS))
keys.head()

441


,case:concept:name,event:id
0,case_000001,evt_00000001
1,case_000001,evt_00000002
2,case_000001,evt_00000003
3,case_000002,evt_00000004
4,case_000002,evt_00000005


In [124]:
# scale with the scaler fitted during training -- fitting a new one here would shift the inputs
scaler = joblib.load("../data/autoencoder/scaler.joblib")
X_scaled = scaler.transform(features.fillna(0).astype("float32").values)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
X_tensor.shape

torch.Size([11017, 441])

In [125]:
# --- The autoencoder (same architecture as in train_autoencoder.ipynb) ---
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

In [126]:
# load the trained weights saved by train_autoencoder.ipynb
checkpoint = torch.load("../data/autoencoder/autoencoder.pt", map_location="cpu")
model = Autoencoder(checkpoint["input_dim"], checkpoint["hidden_dim"], checkpoint["latent_dim"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=441, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=64, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=441, bias=True)
  )
)

In [127]:
# feed ONE event vector through the trained autoencoder
event = X_tensor[0:1]  # first event of case_000001
with torch.no_grad():
    recon, enc = model(event)

print("encoding (latent vector):", np.round(enc.squeeze().numpy(), 3))
print("reconstruction MSE (scaled space):", ((recon - event) ** 2).mean().item())

encoding (latent vector): [ 0.866 -0.134 -0.072  0.361 -0.146 -0.039 -0.171 -0.134  0.026 -0.46
  0.504 -0.69   0.145  0.578 -0.604  0.691 -0.104 -0.311  0.172 -0.491
  0.27  -0.577 -0.495  0.013  0.261 -0.126 -0.494  0.218 -0.433 -1.064
  0.114 -0.47  -0.229  0.621  1.266 -0.543 -0.037 -0.453 -0.573  0.247
 -0.252 -0.657 -0.142 -0.611 -0.548  0.257  0.284  0.439  0.583  0.531
 -0.217 -0.493  0.055 -0.246 -1.34  -0.175  0.045  0.7   -0.927  0.129
  0.776  0.407 -0.309  0.495]
reconstruction MSE (scaled space): 0.049481626600027084


In [128]:
# compare input vs reconstruction in raw feature space (undo the scaling for readability)
comparison = pd.DataFrame({
    "input": features.iloc[0],
    "reconstruction": scaler.inverse_transform(recon.numpy()).squeeze(),
})
comparison["abs_error"] = (comparison["input"] - comparison["reconstruction"]).abs()

print("active (nonzero) features of this event:")
display(comparison[comparison["input"] != 0])

print("largest reconstruction errors:")
comparison.sort_values("abs_error", ascending=False).head(10)

active (nonzero) features of this event:


,input,reconstruction,abs_error
freq_b,1.000000,0.511203,0.488797
seen_b,1.000000,0.982946,0.017054
progress_ratio,0.333333,0.283443,0.049890


largest reconstruction errors:


,input,reconstruction,abs_error
bigram_a->b,0.0,0.572846,0.572846
freq_b,1.0,0.511203,0.488797
seen_a,0.0,0.382270,0.382270
freq_a,0.0,0.188327,0.188327
bigram_b->c,0.0,0.139730,0.139730
seen_l,0.0,-0.099576,0.099576
seen_g,0.0,-0.074769,0.074769
freq_c,0.0,0.063748,0.063748
seen_i,0.0,-0.062683,0.062683
seen_e,0.0,-0.059616,0.059616
